# 04 — Time series with Mapper

`Mapper.Map(value, key, timeseries=True)` appends a Unix timestamp to `key`
and registers it as a *new* entry, instead of overwriting whatever's
currently under that key. Calling it repeatedly on the same base key
accumulates a time series - `AllGather` then returns one entry per tick,
each independently readable.

Real sensor ticks are used below (not a mocked clock) - `int(time.time())`
has 1-second resolution, so ticks are spaced a full second apart to land on
distinct timestamps.

In [ ]:
import os, time
import numpy as np
import pandas as pd
from scarlets.core.Mapper import Mapper
from scarlets.utils.ScarletUtils import redisConnect

os.environ.setdefault("REDIS_HOST", "localhost")
os.environ.setdefault("REDIS_PORT", "6379")
os.environ.setdefault("REDIS_AUTH_TOKEN", "")
os.environ.setdefault("APP_ID", "notebook_worker")

TS_NAME = "tutorial_timeseries"

## Cleanup

In [ ]:
def cleanup():
    mpr = Mapper(TS_NAME)
    mpr.clearAll()

cleanup()
print("cleaned up")

## Simulating sensor ticks

One sensor, five ticks, a second apart, each a small random walk step from
the last reading.

In [ ]:
rng = np.random.default_rng(seed=0)
mpr = Mapper(TS_NAME)

reading = 20.0  # starting temperature
for _ in range(5):
    reading += rng.normal(scale=0.5)
    chunks, ok, exc = mpr.Map(float(reading), "sensorX", timeseries=True)
    print(f"tick: {reading:.2f}  ok={ok}")
    time.sleep(1)

## Reading the series back

`AllGather` returns one dict entry per tick, keyed as `sensorX#<timestamp>`.
Splitting the key on `#` recovers the timestamp each value was written at.

In [ ]:
result, ok, exc = mpr.AllGather()

rows = []
for key, value in result.items():
    base, _, ts = key.partition("#")
    rows.append({"sensor": base, "timestamp": pd.to_datetime(int(ts), unit="s"), "value": value})

series = pd.DataFrame(rows).sort_values("timestamp").set_index("timestamp")
series

## Plotting

In [ ]:
series["value"].plot(marker="o", title="sensorX", ylabel="reading");

## Cleanup (teardown)

In [ ]:
cleanup()
print("cleaned up")